# EN–VI team pilot — Colab
Máy cá nhân không cần GPU. Bật GPU trên Colab; Kaggle cần Internet On.
Qwen dùng NF4/FP16; Gemma dùng NF4/BF16 và cần GPU có native BF16.
Mặc định 2 mẫu × EN/VI; tối đa 10 mẫu mỗi model. Chạy từng model một.
Dùng ZIP mới `pilot_team_v1.zip`, có 30 ảnh pilot đã QA; không dùng ZIP workspace lớn.
Gemma cần tài khoản đã được cấp quyền và secret HF_TOKEN của chính bạn.
Notebook không chứa token hoặc output model. Cloud generation chưa được project xác minh thực tế.

In [ ]:
import os, sys, json, zipfile, shutil, subprocess, hashlib, stat
from pathlib import Path

def safe_extract(archive_path, destination):
    destination=Path(destination).resolve()
    with zipfile.ZipFile(archive_path) as archive:
        if sum(i.file_size for i in archive.infolist())>100*1024**2:
            raise ValueError("Use the small pilot_team ZIP, not the main workspace/weights ZIP")
        names=archive.namelist()
        if len(names)!=len(set(names)):
            raise ValueError("Duplicate archive entries")
        for item in archive.infolist():
            target=(destination/item.filename).resolve()
            if not item.filename.startswith("vlmbench-pilot/") or not target.is_relative_to(destination/"vlmbench-pilot") or ".." in Path(item.filename).parts or stat.S_ISLNK(item.external_attr>>16):
                raise ValueError("Unsafe pilot archive path/type")
        # Do not overwrite a previous workspace silently.
        for item in archive.infolist():
            target=destination/item.filename
            if target.is_file() and target.read_bytes()!=archive.read(item):
                raise ValueError("Existing pilot workspace differs. Use a fresh runtime or separate folder")
        archive.extractall(destination)

PLATFORM="colab"
WORK=Path("/content")
from google.colab import files
uploaded=files.upload()  # Upload exactly one pilot_team_v1.zip.
bundles=[Path(name) for name in uploaded if name.endswith(".zip")]
assert len(bundles)==1, "Upload one current pilot_team ZIP"
safe_extract(bundles[0],WORK)
ROOT=WORK/"vlmbench-pilot"
del uploaded  # Release uploaded ZIP bytes from notebook memory.

assert (ROOT/"BUNDLE_CONTENTS.json").is_file(), "Invalid pilot bundle"
bundle_info=json.loads((ROOT/"BUNDLE_CONTENTS.json").read_text(encoding="utf-8"))
assert bundle_info["manifest"]=="data/pilot/stage1_qa_v1/pilot_manifest.csv", "Rebuild pilot_team_v1.zip with approved pilot manifest"
assert (ROOT/"scripts/run_pilot.py").is_file(), "Old bundle: current launcher is missing"
for name,spec in bundle_info["files"].items():
    target=(ROOT/name).resolve()
    assert target.is_relative_to(ROOT.resolve()), "Invalid bundle index path"
    assert hashlib.sha256(target.read_bytes()).hexdigest()==spec["sha256"], name
print({"project":str(ROOT),"pilot_rows":bundle_info["rows"],"workflow":"team_pilot_v1"})

## 1. Môi trường GPU riêng
Cài đúng runtime lock; cell này chưa tải model weights. Python hỗ trợ 3.11–3.13.
Venv/cache ở /tmp, không lưu vào Drive hoặc Kaggle outputs. Không nâng package tùy ý.

In [ ]:
ENV_DIR=Path("/tmp/vlmbench-infer")
subprocess.run([sys.executable,str(ROOT/"scripts/setup_inference.py"),"--env",str(ENV_DIR)],check=True,cwd=ROOT)
PYTHON=ENV_DIR/"bin/python"
assert PYTHON.is_file()
subprocess.run([str(PYTHON),"-m","vlmbench.cli","doctor"],check=True,cwd=ROOT)

## 2. Chọn model, run name và checkpoint
Mỗi người/phiên dùng RUN_NAME riêng. Lần đầu RESUME=False; tăng LIMIT từ 2 lên 10
hoặc chạy tiếp cùng runtime/config thì đặt RESUME=True. Không nhập checkpoint GPU/config khác.
Gemma cần secret HF_TOKEN được notebook cấp quyền đọc; không paste token vào cell.

In [ ]:
MODEL="qwen"  # "gemma" uses the BF16 runner, never the legacy FP16 adapter.
LIMIT=2
RUN_NAME="smoke_001"  # Change before a new independent run.
RESUME=False
ALLOW_DOWNLOAD=True  # False means cached files only.
os.environ["HF_HOME"]="/tmp/vlmbench-hf-cache"
os.environ["HF_HUB_CACHE"]="/tmp/vlmbench-hf-cache/hub"
if MODEL=="gemma" and ALLOW_DOWNLOAD:
    try:
        from google.colab import userdata
        token=userdata.get("HF_TOKEN")

        if not token:raise ValueError("missing")
        os.environ["HF_TOKEN"]=token
        del token
    except Exception:
        raise RuntimeError("Enable the HF_TOKEN notebook secret for your licensed Gemma account") from None

USE_DRIVE=True  # Mount your own Drive only if you want persistent checkpoint files.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT=Path("/content/drive/MyDrive/vlmbench-results/colab")
else:
    OUT=WORK/"vlmbench-results/colab"

RESULT=OUT/"team_v1"/RUN_NAME/f"{MODEL}.jsonl"

def pilot(*extra):
    argv=[str(PYTHON),str(ROOT/"scripts/run_pilot.py"),"--python",str(PYTHON),
          "--model",MODEL,"--platform",PLATFORM,"--limit",str(LIMIT),
          "--run-name",RUN_NAME,"--output",str(RESULT)]
    if ALLOW_DOWNLOAD:argv.append("--online")
    if RESUME:argv.append("--resume")
    return subprocess.run([*argv,*extra],cwd=ROOT,check=True)

pilot("--dry-run")  # No model download/load, no predictions written.

## 3. Chạy thật trên GPU
Cell dưới tải weights nếu ALLOW_DOWNLOAD=True. GPU check chạy trước tải weights;
Gemma dừng nếu thiếu native BF16, không tự fallback sang FP16/model khác.
Nếu OOM/lỗi, giữ JSONL/meta/summary rồi xem nguyên nhân. Resume chỉ cùng model/config/runtime.
Pilot rows đã human QA approved; đây vẫn là engineering pilot, không phải main benchmark.

In [ ]:
try:
    pilot()
except subprocess.CalledProcessError as error:
    print(f"Pilot stopped with exit code {error.returncode}; inspect the error and keep checkpoint files before retrying.")
summary=RESULT.with_suffix(".summary.json")
if summary.exists():
    print(json.loads(summary.read_text(encoding="utf-8")))

## 4. Tải kết quả
Giữ toàn bộ JSONL/.meta.json/.summary.json/.load_error.json nếu có. ZIP chỉ chứa results,
không chứa token, venv hoặc weights. Tải sau mỗi model, trước khi mất runtime.
Đổi MODEL rồi chạy lại phần 2–4 để thử model còn lại. Không cần chạy lại setup nếu venv còn nguyên.

In [ ]:
assert OUT.exists(), "No output directory yet; inspect pilot/preflight errors"
archive=shutil.make_archive(str(WORK/f"vlmbench-results-{PLATFORM}-{RUN_NAME}"),"zip",root_dir=OUT)
print(archive)
files.download(archive)